In [ ]:
# ============================================================
# STEP 2: Score every sampled frame
# ============================================================
# Process in batches for GPU efficiency.
# For each frame, compute a "Logo Visibility Score" (LVS):
#
#   LVS = player_size_score × 0.40     (biggest factor: large player = visible logo)
#       + jersey_sharpness   × 0.25     (sharp jersey region = readable logo)
#       + frontality_score   × 0.20     (front-facing = chest logos visible)
#       + center_score       × 0.15     (player near center = broadcast focus)
#

def compute_jersey_sharpness(frame, bbox):
    """Compute sharpness of the upper body region (where most logos are)."""
    x1, y1, x2, y2 = [int(v) for v in bbox]
    h_box = y2 - y1
    # Jersey = upper 60% of person bbox (skip head top 15%, skip legs bottom 25%)
    jersey_y1 = y1 + int(h_box * 0.15)
    jersey_y2 = y1 + int(h_box * 0.60)
    jersey_y1 = max(0, jersey_y1)
    jersey_y2 = min(frame.shape[0], jersey_y2)
    x1 = max(0, x1)
    x2 = min(frame.shape[1], x2)

    if jersey_y2 <= jersey_y1 or x2 <= x1:
        return 0.0

    jersey = frame[jersey_y1:jersey_y2, x1:x2]
    gray = cv2.cvtColor(jersey, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def compute_frontality(bbox):
    """Estimate if player is front-facing based on bbox aspect ratio.
    Front-facing: tall and narrower. Side-facing: wider and shorter.
    """
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    if w == 0:
        return 0.0
    aspect = h / w
    # Front-facing players typically have aspect ratio 2.0-3.5
    # Side-running players are wider, aspect ratio 1.0-1.8
    if aspect >= 2.5:
        return 1.0
    elif aspect >= 1.8:
        return 0.7
    elif aspect >= 1.3:
        return 0.4
    else:
        return 0.2


def compute_center_score(bbox, frame_w, frame_h):
    """Score based on how close the player is to frame center."""
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    # Normalized distance from center (0 = center, 1 = corner)
    dx = abs(cx - frame_w / 2) / (frame_w / 2)
    dy = abs(cy - frame_h / 2) / (frame_h / 2)
    dist = (dx**2 + dy**2) ** 0.5 / (2**0.5)
    return 1.0 - dist


def score_frame(frame, detections, frame_w, frame_h):
    """
    Compute Logo Visibility Score for a frame.
    Returns (score, best_player_info) for the most promising player.
    """
    if not detections or len(detections) == 0:
        return 0.0, None

    frame_area = frame_w * frame_h
    best_score = 0.0
    best_info = None

    for det in detections:
        bbox = det[:4]
        conf = det[4]
        x1, y1, x2, y2 = bbox

        # Player size as fraction of frame
        player_area = (x2 - x1) * (y2 - y1)
        size_ratio = player_area / frame_area

        # Skip tiny players (< 1% of frame)
        if size_ratio < 0.01:
            continue

        # Normalize size score (0-1), cap at 25% of frame
        size_score = min(size_ratio / 0.25, 1.0)

        # Jersey sharpness (normalize to 0-1, typical range 0-500)
        sharpness = compute_jersey_sharpness(frame, bbox)
        sharpness_score = min(sharpness / 300.0, 1.0)

        # Frontality
        front_score = compute_frontality(bbox)

        # Center position
        center = compute_center_score(bbox, frame_w, frame_h)

        # Weighted composite score
        lvs = (
            size_score      * 0.40 +
            sharpness_score * 0.25 +
            front_score     * 0.20 +
            center          * 0.15
        )

        if lvs > best_score:
            best_score = lvs
            best_info = {
                "bbox": [float(v) for v in bbox],
                "size_ratio": float(size_ratio),
                "sharpness": float(sharpness),
                "frontality": float(front_score),
                "center": float(center),
                "num_players": len(detections),
            }

    return best_score, best_info


# ============================================================
# MAIN LOOP: Read video → sample → detect → score
# ============================================================
cap = cv2.VideoCapture(str(VIDEO_PATH))

# Add a check to ensure the video was opened successfully
if not cap.isOpened():
    print(f"ERROR: Could not open video file: {VIDEO_PATH}")
    frame_scores = [] # Ensure frame_scores is empty to prevent later errors
else:
    frame_scores = []   # list of (frame_num, timestamp_sec, score, info)
    frame_num = 0
    batch_frames = []
    batch_frame_nums = []

    pbar = tqdm(total=TOTAL_FRAMES, desc="Scanning video", unit="frame")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        pbar.update(1)

        # Temporal sampling
        if frame_num % frame_interval != 0:
            frame_num += 1
            continue

        batch_frames.append(frame)
        batch_frame_nums.append(frame_num)

        # Process batch when full
        if len(batch_frames) >= YOLO_BATCH_SIZE:
            # Batch inference on GPU
            results = model.predict(
                batch_frames,
                classes=[0],  # person only
                conf=0.4,
                device=device,
                verbose=False,
            )

            for i, (fn, fr, res) in enumerate(zip(batch_frame_nums, batch_frames, results)):
                # Extract person detections
                dets = []
                if res.boxes is not None and len(res.boxes) > 0:
                    for j in range(len(res.boxes)):
                        box = res.boxes.xyxy[j].cpu().numpy()
                        conf_val = float(res.boxes.conf[j].cpu().numpy())
                        dets.append([*box, conf_val])

                score, info = score_frame(fr, dets, WIDTH, HEIGHT)

                if score > 0:
                    timestamp = fn / FPS
                    frame_scores.append((fn, timestamp, score, info))

            batch_frames = []
            batch_frame_nums = []

        frame_num += 1

    # Process remaining batch
    if batch_frames:
        results = model.predict(
            batch_frames,
            classes=[0],
            conf=0.4,
            device=device,
            verbose=False,
        )
        for fn, fr, res in zip(batch_frame_nums, batch_frames, results):
            dets = []
            if res.boxes is not None and len(res.boxes) > 0:
                for j in range(len(res.boxes)):
                    box = res.boxes.xyxy[j].cpu().numpy()
                    conf_val = float(res.boxes.conf[j].cpu().numpy())
                    dets.append([*box, conf_val])
            score, info = score_frame(fr, dets, WIDTH, HEIGHT)
            if score > 0:
                timestamp = fn / FPS
                frame_scores.append((fn, timestamp, score, info))

    pbar.close()

cap.release()

print(f"\nScored {len(frame_scores):,} frames with players")
# Add a check to prevent ValueError if frame_scores is empty
if frame_scores:
    print(f"Score range: {min(s[2] for s in frame_scores):.3f} — {max(s[2] for s in frame_scores):.3f}")
else:
    print("No frames were scored. This might indicate an issue with video reading, person detection, or scoring logic.")